# Summarization Examples

Map-reduce summarization of a long text (Moby-Dick): splits the text into token-based chunks, summarizes each chunk with Claude, then combines the chunk summaries into one final summary.

In [ ]:
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
import getpass

load_dotenv()

In [4]:
llm = ChatAnthropic(model="claude-haiku-4-5")

In [7]:
text_chunks_chain = (
    RunnableLambda(lambda x: 
        [
            {
                'chunk': text_chunk, 
            }
            for text_chunk in
               TokenTextSplitter(chunk_size=3000,
                                 chunk_overlap=100).split_text(x)
        ]
    )
)

In [9]:
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt =PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }
    )
)

In [10]:
summarize_summaries_prompt_template = """
Write a concise summary of the following text, 
which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt =PromptTemplate.from_template(summarize_summaries_prompt_template)

summarize_reduce_chain = (
    RunnableLambda(lambda x: 
        {
            'summaries': '\n'.join([i['summary'] for i in x]), 
        })
    | summarize_summaries_prompt 
    | llm 
    | StrOutputParser()
)

In [11]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)

In [12]:
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [13]:
summary = map_reduce_chain.invoke(moby_dick_book)